In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
df = pd.read_csv("../../data/diabetes_binary_classification_data.csv")

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
display(df.describe().T)

for col in df.select_dtypes(["object", "category"]):
    print(f"\n--- {col} ({df[col].nunique()} unique) ---")
    print(df[col].value_counts().to_string())

for col in df.select_dtypes("object"):
    cleaned = df[col].str.strip().str.lower()
    if cleaned.nunique() < df[col].nunique():
        print(f"{col}: case/whitespace variants found")

In [ ]:
counts = df["Diabetes_binary"].value_counts().sort_index()
counts.index = ["No diabetes", "Prediabetes/diabetes"]
rate = counts / counts.sum() * 100

plt.figure(figsize=(8, 5))
ax = sns.barplot(x=rate.index, y=rate.values, width=0.4, saturation=1, edgecolor="black")

ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
ax.set_xticks(range(len(counts)))
ax.set_xticklabels([f"{label}\n(n = {n:,})" for label, n in counts.items()])
ax.set(xlabel="Diabetes status", ylabel="Respondents (%)")
plt.title("Distribution of Diabetes Classes", fontweight="bold")
plt.show()

In [ ]:
def plot_rate(col, labels, xlabel, title, figsize=(8, 5)):
    stats = df.groupby(col)["Diabetes_binary"].agg(rate="mean", n="size")
    stats["rate"] *= 100
    stats.index = labels

    plt.figure(figsize=figsize)
    ax = sns.barplot(x=stats.index, y=stats["rate"], width=0.4, saturation=1, edgecolor="black")

    ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
    ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
    ax.set_xticks(range(len(stats)))
    ax.set_xticklabels([f"{label}\n(n = {n:,})" for label, n in stats["n"].items()])
    ax.set_ylim(0, stats["rate"].max() * 1.1)
    ax.set(xlabel=xlabel, ylabel="Diabetes rate (%)")
    plt.title(title, fontweight="bold")
    plt.show()


plot_rate("HighBP", ["No high BP", "High BP"], "Blood pressure status", "Diabetes Rate by High Blood Pressure")
plot_rate("HighChol", ["No high chol", "High chol"], "Cholesterol status", "Diabetes Rate by High Cholesterol")
plot_rate("Sex", ["Female", "Male"], "Sex", "Diabetes Rate by Sex")

In [ ]:
status = df["Diabetes_binary"].map({0: "No diabetes", 1: "Prediabetes/diabetes"})
order = ["No diabetes", "Prediabetes/diabetes"]

plt.figure(figsize=(12, 4))
sns.boxplot(x=df["BMI"], y=status, order=order, width=0.3, saturation=1,
            boxprops={"edgecolor": "black"},
            flierprops={"markersize": 2.5, "markerfacecolor": "red",
                        "markeredgecolor": "black", "alpha": 0.25})

for i, median in enumerate(df.groupby(status)["BMI"].median().reindex(order)):
    plt.text(median, i - 0.25, f"Median: {median:.0f}", ha="center")

plt.xlim(0, 100)
plt.title("BMI by Diabetes Status", fontweight="bold")
plt.xlabel("BMI")
plt.ylabel("")
plt.show()

In [ ]:
age_labels = {1: "18–24", 2: "25–29", 3: "30–34", 4: "35–39", 5: "40–44", 6: "45–49", 7: "50–54",
              8: "55–59", 9: "60–64", 10: "65–69", 11: "70–74", 12: "75–79", 13: "80+"}

# Total
counts = df["Age"].value_counts().sort_index().rename(age_labels)

plt.figure(figsize=(14, 6))
ax = sns.barplot(x=counts.index, y=counts.values, saturation=1, edgecolor="black")

ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax.bar_label(ax.containers[0], fmt="{:,.0f}", padding=3, fontsize=9)
ax.set(xlabel="Age Group", ylabel="Number of People")
plt.title("Number of People by Age Group", fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.show()

# Percentage
rate = df.groupby("Age")["Diabetes_binary"].mean().mul(100).rename(age_labels)

plt.figure(figsize=(14, 6))
ax = sns.barplot(x=rate.index, y=rate.values, saturation=1, edgecolor="black")

ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3, fontsize=9)
ax.set(xlabel="Age Group", ylabel="Has Diabetes (%)")
plt.title("Diabetes Rate by Age", fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
stats = df.groupby("GenHlth")["Diabetes_binary"].agg(rate="mean", n="size")
stats["rate"] *= 100
stats.index = ["Excellent", "Very good", "Good", "Fair", "Poor"]

plt.figure(figsize=(10, 5))
ax = sns.barplot(x=stats.index, y=stats["rate"], width=0.6, saturation=1, edgecolor="black")

ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
ax.set_xticks(range(len(stats)))
ax.set_xticklabels([f"{label}\n(n = {n:,})" for label, n in stats["n"].items()])
ax.set_ylim(0, stats["rate"].max() * 1.1)
ax.set(xlabel="General health", ylabel="Diabetes rate (%)")
plt.title("Diabetes Rate by General Health", fontweight="bold")
plt.show()

In [ ]:
status = df["Diabetes_binary"].map({0: "No diabetes", 1: "Prediabetes/diabetes"})
order = ["No diabetes", "Prediabetes/diabetes"]

plt.figure(figsize=(12, 4))
sns.boxplot(x=df["PhysHlth"], y=status, order=order, width=0.3, saturation=1,
            boxprops={"edgecolor": "black"},
            flierprops={"markersize": 2.5, "markerfacecolor": "red",
                        "markeredgecolor": "black", "alpha": 0.25})

for i, median in enumerate(df.groupby(status)["PhysHlth"].median().reindex(order)):
    plt.text(median, i - 0.25, f"Median: {median:.0f}", ha="center")

plt.title("Physical Health by Diabetes Status", fontweight="bold")
plt.xlabel("Poor physical health days (past 30 days)")
plt.ylabel("")
plt.show()

In [ ]:
plt.figure(figsize=(20, 15))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="Spectral",
            center=0, linewidths=0.5, linecolor="black")
plt.title("Correlation of the whole dataframe", fontweight="bold")
plt.show()

In [ ]:
target_to_compare_to = df.corr(numeric_only=True)["Diabetes_binary"].sort_values(ascending=False)

plt.figure(figsize=(3, 10))
sns.heatmap(target_to_compare_to.to_frame(), cmap="Spectral", annot=True, fmt=".2f", cbar=True)
plt.title(f"Correlation with {"Diabetes_binary"}", fontweight="bold")
plt.show()

In [ ]:
target_to_compare_to = df.corr(numeric_only=True)["GenHlth"].sort_values(ascending=False)

plt.figure(figsize=(3, 10))
sns.heatmap(target_to_compare_to.to_frame(), cmap="Spectral", annot=True, fmt=".2f", cbar=True)
plt.title(f"Correlation with {"GenHlth"}", fontweight="bold")
plt.show()

In [ ]:
target_to_compare_to = df.corr(numeric_only=True)["Income"].sort_values(ascending=False)

plt.figure(figsize=(3, 10))
sns.heatmap(target_to_compare_to.to_frame(), cmap="Spectral", annot=True, fmt=".2f", cbar=True)
plt.title(f"Correlation with {"Income"}", fontweight="bold")
plt.show()